# BirdCLEF 2026 - Training Notebook with Checkpoint Resuming

## Overview
This notebook trains a baseline CNN model for multi-label audio classification of Pantanal wildlife species.

**Key Features:**
- Mel-spectrogram feature extraction
- EfficientNet-B0 backbone
- Multi-label classification (234 species)
- Audio augmentation
- Cross-validation
- **✨ Automatic checkpoint saving and resuming**

**Competition:** https://www.kaggle.com/competitions/birdclef-2026

In [ ]:
# Install compatible versions
!pip install -q numpy==1.26.4 scipy

In [ ]:
# Install PyTorch 2.2.0 (compatible with Tesla P100)
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.2.0+cu118 torchvision==0.17.0+cu118 torchaudio==2.2.0+cu118 --index-url https://download.pytorch.org/whl/cu118

# Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install -q timm librosa soundfile audiomentations

In [ ]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Audio processing
import librosa
import soundfile as sf
import audiomentations as AA

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

# Sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
class CFG:
    # Paths
    data_dir = Path('/kaggle/input/birdclef-2026')
    train_audio_dir = data_dir / 'train_audio'
    train_soundscapes_dir = data_dir / 'train_soundscapes'
    output_dir = Path('/kaggle/working')
    
    # Audio parameters
    sample_rate = 32000
    duration = 5  # seconds
    n_mels = 128
    fmin = 20
    fmax = 16000
    n_fft = 2048
    hop_length = 512
    
    # Model parameters
    model_name = 'efficientnet_b0'
    pretrained = True
    num_classes = 234
    
    # Training parameters
    n_folds = 5
    train_folds = [0, 1, 2, 3]  # Train on 4 folds, validate on 1
    seed = 42
    epochs = 10
    batch_size = 32
    lr = 1e-3
    weight_decay = 1e-6
    num_workers = 2
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Augmentation
    use_augmentation = True
    
    # Checkpoint settings
    save_checkpoint_every_epoch = True
    auto_resume = True  # Automatically resume from checkpoint if exists
    
CFG.output_dir.mkdir(exist_ok=True, parents=True)
print(f"Device: {CFG.device}")
print(f"Auto-resume enabled: {CFG.auto_resume}")

## 3. Seed Everything

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.seed)

## 4. Load and Prepare Data

In [ ]:
# Load metadata
train_df = pd.read_csv(CFG.data_dir / 'train.csv')
taxonomy_df = pd.read_csv(CFG.data_dir / 'taxonomy.csv')

print(f"Train samples: {len(train_df)}")
print(f"Number of species: {len(taxonomy_df)}")
print(f"\nTrain columns: {train_df.columns.tolist()}")

# Display sample
train_df.head()

In [ ]:
# Get all unique species labels
species_list = taxonomy_df['primary_label'].tolist()
species_to_idx = {species: idx for idx, species in enumerate(species_list)}
idx_to_species = {idx: species for species, idx in species_to_idx.items()}

print(f"Total species: {len(species_list)}")
print(f"First 10 species: {species_list[:10]}")

In [ ]:
# Create multi-label targets
def create_multilabel_target(row, species_to_idx):
    """Create multi-label target vector"""
    target = np.zeros(len(species_to_idx), dtype=np.float32)
    
    # Primary label
    if row['primary_label'] in species_to_idx:
        target[species_to_idx[row['primary_label']]] = 1.0
    
    # Secondary labels
    if pd.notna(row['secondary_labels']):
        secondary = str(row['secondary_labels']).strip('[]').replace("'", "").split(', ')
        for label in secondary:
            label = label.strip()
            if label and label in species_to_idx:
                target[species_to_idx[label]] = 1.0
    
    return target

# Apply to dataframe
train_df['target'] = train_df.apply(lambda row: create_multilabel_target(row, species_to_idx), axis=1)

# Check distribution
targets_array = np.stack(train_df['target'].values)
print(f"Target shape: {targets_array.shape}")
print(f"Average labels per sample: {targets_array.sum(axis=1).mean():.2f}")
print(f"Samples per species (top 10): {targets_array.sum(axis=0).argsort()[::-1][:10]}")

## 5. Create Folds

In [ ]:
# Stratified K-Fold based on primary label
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train_df['fold'] = -1

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    train_df.loc[val_idx, 'fold'] = fold

print("Fold distribution:")
print(train_df['fold'].value_counts().sort_index())

## 6. Audio Processing Functions

In [ ]:
def load_audio(filepath, duration=5, sr=32000):
    """Load audio file and ensure it's the correct duration"""
    try:
        # Load audio
        audio, orig_sr = sf.read(filepath)
        
        # Resample if needed
        if orig_sr != sr:
            audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sr)
        
        # Convert to mono if stereo
        if len(audio.shape) > 1:
            audio = audio.mean(axis=1)
        
        # Ensure correct duration
        target_length = sr * duration
        
        if len(audio) > target_length:
            # Random crop
            start = np.random.randint(0, len(audio) - target_length)
            audio = audio[start:start + target_length]
        elif len(audio) < target_length:
            # Pad with zeros
            audio = np.pad(audio, (0, target_length - len(audio)), mode='constant')
        
        return audio.astype(np.float32)
    
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        # Return silence if error
        return np.zeros(sr * duration, dtype=np.float32)


def audio_to_melspectrogram(audio, sr=32000, n_mels=128, fmin=20, fmax=16000, n_fft=2048, hop_length=512):
    """Convert audio to mel-spectrogram"""
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax,
        n_fft=n_fft,
        hop_length=hop_length
    )
    
    # Convert to dB scale
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize to [0, 1]
    mel_spec_norm = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
    
    return mel_spec_norm.astype(np.float32)

## 7. Audio Augmentation

In [ ]:
# Audio augmentation pipeline
train_augmentation = AA.Compose([
    AA.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    AA.TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5),
    AA.PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    AA.Shift(min_shift=-0.5, max_shift=0.5, p=0.5),
])

print("Augmentation pipeline created")

## 8. Dataset Class

In [ ]:
class BirdCLEFDataset(Dataset):
    def __init__(self, df, audio_dir, species_to_idx, cfg, augmentation=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.species_to_idx = species_to_idx
        self.cfg = cfg
        self.augmentation = augmentation
        self.is_train = is_train
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load audio
        filepath = self.audio_dir / row['filename']
        audio = load_audio(filepath, duration=self.cfg.duration, sr=self.cfg.sample_rate)
        
        # Apply augmentation
        if self.is_train and self.augmentation is not None:
            audio = self.augmentation(samples=audio, sample_rate=self.cfg.sample_rate)
        
        # Convert to mel-spectrogram
        mel_spec = audio_to_melspectrogram(
            audio,
            sr=self.cfg.sample_rate,
            n_mels=self.cfg.n_mels,
            fmin=self.cfg.fmin,
            fmax=self.cfg.fmax,
            n_fft=self.cfg.n_fft,
            hop_length=self.cfg.hop_length
        )
        
        # Convert to 3-channel image (duplicate channels for pretrained models)
        mel_spec = np.stack([mel_spec, mel_spec, mel_spec], axis=0)
        
        # Get target
        target = row['target']
        
        return {
            'image': torch.tensor(mel_spec, dtype=torch.float32),
            'target': torch.tensor(target, dtype=torch.float32)
        }

## 9. Model Definition

In [ ]:
class BirdCLEFModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=234, pretrained=True):
        super().__init__()
        
        # Load pretrained backbone
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,  # Remove classification head
            global_pool=''
        )
        
        # Get number of features
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 128, 313)  # Approximate mel-spec shape
            features = self.backbone(dummy_input)
            n_features = features.shape[1]
        
        # Global pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, num_classes)
        )
    
    def forward(self, x):
        # Extract features
        features = self.backbone(x)
        
        # Global pooling
        pooled = self.global_pool(features)
        pooled = pooled.view(pooled.size(0), -1)
        
        # Classification
        output = self.classifier(pooled)
        
        return output

# Test model
model = BirdCLEFModel(CFG.model_name, CFG.num_classes, CFG.pretrained)
print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

## 10. Checkpoint Functions

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_score, fold, cfg):
    """Save training checkpoint"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_score': best_score,
        'fold': fold,
        'config': {
            'model_name': cfg.model_name,
            'num_classes': cfg.num_classes,
            'lr': cfg.lr,
            'epochs': cfg.epochs
        }
    }
    checkpoint_path = cfg.output_dir / f'checkpoint_fold{fold}.pth'
    torch.save(checkpoint, checkpoint_path)
    print(f"💾 Checkpoint saved at epoch {epoch+1}")


def load_checkpoint(model, optimizer, scheduler, fold, cfg):
    """Load training checkpoint if exists"""
    checkpoint_path = cfg.output_dir / f'checkpoint_fold{fold}.pth'
    
    if checkpoint_path.exists():
        try:
            checkpoint = torch.load(checkpoint_path, map_location=cfg.device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_score = checkpoint['best_score']
            
            print(f"\n{'='*60}")
            print(f"🔄 RESUMING FROM CHECKPOINT")
            print(f"{'='*60}")
            print(f"Fold: {fold}")
            print(f"Resuming from epoch: {start_epoch}")
            print(f"Best score so far: {best_score:.4f}")
            print(f"Remaining epochs: {cfg.epochs - start_epoch}")
            print(f"{'='*60}\n")
            
            return start_epoch, best_score
        except Exception as e:
            print(f"⚠️  Error loading checkpoint: {e}")
            print("Starting training from scratch...")
            return 0, 0.0
    
    print(f"No checkpoint found for fold {fold}. Starting from scratch.")
    return 0, 0.0


def delete_checkpoint(fold, cfg):
    """Delete checkpoint after successful training completion"""
    checkpoint_path = cfg.output_dir / f'checkpoint_fold{fold}.pth'
    if checkpoint_path.exists():
        checkpoint_path.unlink()
        print(f"🗑️  Deleted checkpoint for fold {fold} (training completed)")

## 11. Training Functions

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Train]")
    for batch in pbar:
        images = batch['image'].to(device)
        targets = batch['target'].to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, targets)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': running_loss / (pbar.n + 1)})
    
    return running_loss / len(dataloader)


def validate_one_epoch(model, dataloader, criterion, device, epoch):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Valid]")
    with torch.no_grad():
        for batch in pbar:
            images = batch['image'].to(device)
            targets = batch['target'].to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item()
            
            # Store predictions and targets
            preds = torch.sigmoid(outputs).detach().cpu().numpy()
            all_preds.append(preds)
            all_targets.append(targets.detach().cpu().numpy())
            
            pbar.set_postfix({'loss': running_loss / (pbar.n + 1)})
    
    # Calculate metrics
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    
    # Macro ROC-AUC (skip classes with no positive labels)
    valid_classes = all_targets.sum(axis=0) > 0
    if valid_classes.sum() > 0:
        score = roc_auc_score(
            all_targets[:, valid_classes],
            all_preds[:, valid_classes],
            average='macro'
        )
    else:
        score = 0.0
    
    return running_loss / len(dataloader), score

## 12. Training Loop with Checkpoint Support

In [ ]:
def train_fold(fold, train_df, cfg):
    print(f"\n{'='*60}")
    print(f"Training Fold {fold}")
    print(f"{'='*60}\n")
    
    # Split data
    train_data = train_df[train_df['fold'] != fold].reset_index(drop=True)
    valid_data = train_df[train_df['fold'] == fold].reset_index(drop=True)
    
    print(f"Train samples: {len(train_data)}")
    print(f"Valid samples: {len(valid_data)}")
    
    # Create datasets
    train_dataset = BirdCLEFDataset(
        train_data,
        cfg.train_audio_dir,
        species_to_idx,
        cfg,
        augmentation=train_augmentation if cfg.use_augmentation else None,
        is_train=True
    )
    
    valid_dataset = BirdCLEFDataset(
        valid_data,
        cfg.train_audio_dir,
        species_to_idx,
        cfg,
        augmentation=None,
        is_train=False
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True
    )
    
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True
    )
    
    # Create model
    model = BirdCLEFModel(cfg.model_name, cfg.num_classes, cfg.pretrained)
    model = model.to(cfg.device)
    
    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    
    # Try to resume from checkpoint
    start_epoch = 0
    best_score = 0.0
    
    if cfg.auto_resume:
        start_epoch, best_score = load_checkpoint(model, optimizer, scheduler, fold, cfg)
    
    # Training loop
    for epoch in range(start_epoch, cfg.epochs):
        # Train
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, cfg.device, epoch)
        
        # Validate
        valid_loss, valid_score = validate_one_epoch(model, valid_loader, criterion, cfg.device, epoch)
        
        # Step scheduler
        scheduler.step()
        
        print(f"\n{'─'*60}")
        print(f"Epoch {epoch+1}/{cfg.epochs} Summary")
        print(f"{'─'*60}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Valid Loss: {valid_loss:.4f}")
        print(f"Valid ROC-AUC: {valid_score:.4f}")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save best model
        if valid_score > best_score:
            best_score = valid_score
            torch.save(
                model.state_dict(),
                cfg.output_dir / f'best_model_fold{fold}.pth'
            )
            print(f"⭐ New best model saved! (score: {best_score:.4f})")
        else:
            print(f"Best score: {best_score:.4f}")
        
        # Save checkpoint
        if cfg.save_checkpoint_every_epoch:
            save_checkpoint(model, optimizer, scheduler, epoch, best_score, fold, cfg)
        
        print(f"{'─'*60}\n")
        
        # Clean up
        gc.collect()
        torch.cuda.empty_cache()
    
    # Training completed - delete checkpoint
    print(f"\n✅ Fold {fold} training completed!")
    delete_checkpoint(fold, cfg)
    
    return best_score

## 13. Train All Folds

In [ ]:
# Train specified folds
fold_scores = []

for fold in CFG.train_folds:
    score = train_fold(fold, train_df, CFG)
    fold_scores.append(score)
    print(f"\n{'='*60}")
    print(f"Fold {fold} Best Score: {score:.4f}")
    print(f"{'='*60}\n")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Average CV Score: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"Individual Fold Scores:")
for i, score in enumerate(fold_scores):
    print(f"  Fold {CFG.train_folds[i]}: {score:.4f}")
print(f"{'='*60}")

## 14. Save Training Metadata

In [ ]:
# Save species mapping
import json

metadata = {
    'species_to_idx': species_to_idx,
    'idx_to_species': idx_to_species,
    'fold_scores': {f'fold_{i}': score for i, score in enumerate(fold_scores)},
    'avg_cv_score': float(np.mean(fold_scores)),
    'std_cv_score': float(np.std(fold_scores)),
    'config': {
        'model_name': CFG.model_name,
        'sample_rate': CFG.sample_rate,
        'duration': CFG.duration,
        'n_mels': CFG.n_mels,
        'num_classes': CFG.num_classes,
        'epochs': CFG.epochs,
        'batch_size': CFG.batch_size,
        'lr': CFG.lr
    }
}

with open(CFG.output_dir / 'training_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Training metadata saved!")

## Summary

This training notebook with checkpoint support:
- ✅ Loads and preprocesses audio data
- ✅ Creates mel-spectrogram features
- ✅ Implements audio augmentation
- ✅ Trains EfficientNet-B0 model
- ✅ Uses cross-validation
- ✅ **Automatically saves checkpoints every epoch**
- ✅ **Automatically resumes from last checkpoint if interrupted**
- ✅ Saves best models for inference

### Checkpoint Features:

**Automatic Resuming:**
- If training is interrupted, simply re-run the notebook
- It will automatically detect and load the last checkpoint
- Training continues from where it left off
- No manual intervention needed!

**What's Saved:**
- Model weights
- Optimizer state
- Scheduler state
- Current epoch
- Best validation score
- Configuration

**Checkpoint Management:**
- Checkpoints are saved after each epoch
- Automatically deleted when fold training completes
- Separate checkpoint for each fold

**Next Steps:**
1. Use the inference notebook to make predictions
2. Try advanced models (EfficientNet-B2, PANNs)
3. Add more augmentation techniques
4. Ensemble multiple models
5. Use pseudo-labeling on unlabeled soundscapes